# Homework Assignment Solution: Data Preprocessing

BINF 6210/8210: Machine Learning for Bioinformatics
___

Complete every TODO. Keep outputs visible and add short written interpretations._

This notebook provides one defensible solution. Alternative scientifically justified choices may receive full credit.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, cross_validate
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, RobustScaler, OneHotEncoder, FunctionTransformer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, ConfusionMatrixDisplay
RANDOM_STATE = 42
rng = np.random.default_rng(RANDOM_STATE)

In [2]:
n = 240

df = pd.DataFrame({
    "sample_id": [f"S{i:03d}" for i in range(n)],
    "age": rng.normal(52, 14, n).clip(18, 85),
    "bmi": rng.normal(27, 5, n).clip(16, 48),
    "batch": rng.choice(["B1", "B2", "B3"], n, p=[.45,.35,.20]),
    "sex": rng.choice(["Female", "Male"], n),
    "metabolite_a": rng.lognormal(1.2, .8, n),
    "metabolite_b": rng.lognormal(.5, .6, n),
})

logit = -3 + .045*df.age + .7*np.log1p(df.metabolite_a) + .35*(df.sex=="Male")
df["case"] = rng.binomial(1, 1/(1+np.exp(-logit)))

for col, frac in {"bmi":.08,"metabolite_a":.12,"batch":.04}.items():
    df.loc[rng.choice(n, int(n*frac), replace=False), col] = np.nan

df.loc[5, "bmi"] = 120  # deliberate data-quality issue

df.head()

,sample_id,age,bmi,batch,sex,metabolite_a,metabolite_b,case
0,S000,56.266039,NaN,B2,Female,3.021977,2.166286,1
1,S001,37.440223,26.528687,B2,Female,NaN,3.923859,1
2,S002,62.506317,18.211358,B3,Female,0.391305,1.573947,1
3,S003,65.167906,19.664774,B3,Female,4.557376,1.465020,1
4,S004,24.685507,37.646236,B2,Female,11.578507,0.844703,1


## Part A - Audit (20 points)
1. State the observational unit, target, identifier, numeric features, and categorical features.
2. Produce a compact quality report containing dtype, unique count, and missing fraction.
3. Identify the deliberate implausible value and propose a defensible handling rule.

In [3]:
report = pd.DataFrame({"dtype": df.dtypes.astype(str),
                       "unique": df.nunique(dropna=False),
                       "missing_fraction":df.isna().mean()})
display(report)

print("Observational unit: one sample; target: case; identifier: sample_id")
print("The BMI=120 record requires source verification; do not silently delete it.")

,dtype,unique,missing_fraction
sample_id,str,240,0.000000
age,float64,239,0.000000
bmi,float64,219,0.079167
batch,str,4,0.037500
sex,str,2,0.000000
metabolite_a,float64,213,0.116667
metabolite_b,float64,240,0.000000
case,int64,2,0.000000


Observational unit: one sample; target: case; identifier: sample_id
The BMI=120 record requires source verification; do not silently delete it.


## Part B - Missingness and splitting (30 points)
4. Compare missing fractions of bmi, metabolite_a, and batch by outcome class.
5. Create a 75/25 (testing data accounting for 25% of the data) stratified split with `random_state=42`. Verify the class proportion in both sets.


In [14]:
print('Missing fraction by outcome class:')
display(df.groupby("case").agg({c:lambda s:s.isna().mean() for c in ["bmi", "metabolite_a", "batch"]}))

X = df.drop(columns=["case","sample_id"]);
y = df.case

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=.25, stratify=y, random_state=RANDOM_STATE)

print('\ny_train proportion:', y_train.shape[0] / y.shape[0])
print('y_test proportion:', y_test.shape[0] / y.shape[0])

class_proportion_df = pd.DataFrame(index=['case 0', 'case 1'], columns=['y', 'y_train', 'y_test'])
class_proportion_df.loc['case 1', 'y'] = sum(y)/len(y)
class_proportion_df.loc['case 0', 'y'] = 1 - sum(y)/len(y)
class_proportion_df.loc['case 1', 'y_train'] = sum(y_train)/len(y_train)
class_proportion_df.loc['case 0', 'y_train'] = 1 - sum(y_train)/len(y_train)
class_proportion_df.loc['case 1', 'y_test'] = sum(y_test)/len(y_test)
class_proportion_df.loc['case 0', 'y_test'] = 1 - sum(y_test)/len(y_test)

print('\nClass proportion in y_train and y_test:')
print(class_proportion_df)

Missing fraction by outcome class:


,bmi,metabolite_a,batch
case,,,
0,0.107843,0.107843,0.009804
1,0.057971,0.123188,0.057971



y_train proportion: 0.75
y_test proportion: 0.25

Class proportion in y_train and y_test:
            y   y_train    y_test
case 0  0.425  0.427778  0.416667
case 1  0.575  0.572222  0.583333


### Part B Report

The training and testing dataset account for 75% and 25% of the original data, as expected.

The distribution of case 0 and case 1 remain consistent in y_train and y_test.

## Part C - Leakage-safe preprocessing (40 points)
6. Build a `ColumnTransformer`: numeric median imputation + `StandardScaler`; categorical most-frequent imputation + `OneHotEncoder(handle_unknown="ignore")`.
7. Fit it only on training data. Report transformed shapes and feature names.
8. Explain why fitting the transformer before splitting would bias evaluation.

In [17]:
numerical_variables = ["age", "bmi", "metabolite_a", "metabolite_b"];
categorical_variables = ["batch", "sex"]

numerical_data_pipeline = Pipeline([
    ("impute", SimpleImputer(strategy="median")),
    ("scale", StandardScaler())
])

categorical_data_pipeline = Pipeline([
    ("impute", SimpleImputer(strategy="most_frequent")),
    ("encode", OneHotEncoder(handle_unknown="ignore"))
])

preprocess = ColumnTransformer([
    ("numerical", numerical_data_pipeline, numerical_variables),
    ("categorical",categorical_data_pipeline, categorical_variables)])

X_train_transformed = preprocess.fit_transform(X_train);
X_test_transformed = preprocess.transform(X_test)

print('X_train transformed shape and X_test_transformed shape:')
print(X_train_transformed.shape, X_test_transformed.shape);

print('\nFeature names:')
print(preprocess.get_feature_names_out())

X_train transformed shape and X_test_transformed shape:
(180, 9) (60, 9)

Feature names:
['numerical__age' 'numerical__bmi' 'numerical__metabolite_a'
 'numerical__metabolite_b' 'categorical__batch_B1' 'categorical__batch_B2'
 'categorical__batch_B3' 'categorical__sex_Female' 'categorical__sex_Male']


In [ ]:
# TODO


## Part D - Reproducibility (10 points)

List the software versions used and explain what the random seed does - and does not - guarantee.


In [19]:
import sys, sklearn

print(sys.version)
print("pandas: ", pd.__version__, "; numpy: ", np.__version__, "; scikit-learn: ",sklearn.__version__)

3.14.2 (main, Dec  5 2025, 16:49:16) [Clang 17.0.0 (clang-1700.6.3.2)]
pandas:  3.0.5 ; numpy:  2.5.2 ; scikit-learn:  1.9.0


A random seed controls a pseudorandom stream; it does not guarantee identical results across all hardware, library versions, or parallel algorithms.